# encoder only 
 Input Text-> Encoder -> Representation -> Classification
 These Encoder Only Models read entire input at once and their common uses 
  1. Sentiment classification
  2. Spam detection
  3. Named-entity recognition
  4. Semantic similarity
  5. Search embeddings
  6. Question answering

In [ ]:
# Sentiment Classification Example
import torch
print(f"torch version {torch.__version__}")
from transformers import  (AutoTokenizer, AutoModelForSequenceClassification)

model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
#model_id = "VijayKanupuri/bert-sst2"


tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

text = "Hugging face is easy to learn"


input_text_token = tokenizer(text,return_tensors="pt")

with torch.no_grad():
    outputs = model(**input_text_token) # ** is for unpacking dictionary and * is for unpacking a list or tuple 


prob = torch.softmax(outputs.logits,dim=-1)
print(prob)
predicted_id = prob.argmax(dim=-1).item()

print(model.config.id2label[predicted_id])

# confidence = prob[0][predicted_id].item()

# print(predicted_id, confidence)

torch version 2.5.1+cu121


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 190.57it/s]


tensor([[0.0065, 0.9935]])
LABEL_1
1 0.9934946894645691


# spam detection 

In [27]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = "mrm8488/bert-tiny-finetuned-sms-spam-detection"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

text = "Congratulations! You have won a free prize. Claim now!"

input_text_token = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**input_text_token)

prob = torch.softmax(outputs.logits, dim=-1)

print("Probabilities:", prob)

predicted_id = prob.argmax(dim=-1).item()

print("Predicted ID:", predicted_id)
print("Prediction:", model.config.id2label[predicted_id])

Loading weights: 100%|██████████| 41/41 [00:00<00:00, 5987.06it/s]

Probabilities: tensor([[0.9353, 0.0647]])
Predicted ID: 0
Prediction: LABEL_0


## Named Entity Recognition 

In [30]:
import torch

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer
)

model_id = "dslim/bert-base-NER"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForTokenClassification.from_pretrained(model_id)

text = "I live in India but I want to visit Switzerland once in life"

input_text_token = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**input_text_token)

print("Logits shape:", outputs.logits.shape)

predicted_ids = outputs.logits.argmax(dim=-1)

print("Predicted IDs:", predicted_ids)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 455.53it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Logits shape: torch.Size([1, 15, 9])
Predicted IDs: tensor([[0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0]])


In [31]:
print(tokenizer.convert_ids_to_tokens(input_text_token["input_ids"][0]))

['[CLS]', 'I', 'live', 'in', 'India', 'but', 'I', 'want', 'to', 'visit', 'Switzerland', 'once', 'in', 'life', '[SEP]']


# Generative Question Answer (without Context)

In [ ]:
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

#question = "Answer the following question correctly: What is the capital of India?"
question = """
Context: India is a country in South Asia.
The capital city of India is New Delhi.

Question: What is the capital of Pakistan?
Answer:
"""

inputs = tokenizer(question,return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10
    )

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", answer)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 619.02it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer: New Delhi
